In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from sklearn.metrics import silhouette_score
import os
import bg_logger
from sklearn.decomposition import PCA

In [ ]:
# 0. Plotting details and create new directory

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.linewidth": 1.2,
    "lines.linewidth": 1.5,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "legend.frameon": False,
})

new_dir = "outputs"
if not os.path.exists(new_dir):
    os.mkdir(new_dir)

logger = bg_logger.BG_logging(stage_name="BG_ML", log_file=os.path.join(new_dir, "bg_ML.log")).setup_logger()

In [ ]:

# 1. Load
df = pd.read_csv("ml_features.csv")
logger.info("Loading .csv file from bg_main.py")

# 2. Drop/flag bad rows
df = df[df["score"].notna() & np.isfinite(df["sigma"])]
logger.info("Dropping rows with invalid score or sigma from lc_flagging")

# 3. Features to use (drop gaia_id, filter — not numeric)
features = [
    "score", "n_dips", "sigma", "survival_fraction", "n_high_cut",
    "p2p_over_mad", "duty_cycle", "spacing_frac_scatter", "consistent_spacings",
    "best_depth_flux", "best_depth_frac", "best_depth_sigma",
    "best_mf_snr", "best_duration", "best_n_points",
]
X = df[features]
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=features)
logger.info("Identified lc_flagging features -- scaled using StandardScaler()")


In [ ]:
def rms(x):
    return np.sqrt(np.mean(x**2))

In [ ]:
# 4. Plot the histograms for each scaled data
logger.info("Plotting a histogram for each of the scaled features")
n_bins = 10 
fig, ax = plt.subplots(len(features), 1, figsize=(10,10))
for i, ft in enumerate(features):
    ax[i].hist(X_scaled[ft], bins=n_bins)
    ax[i].set_ylabel("Counts")
    ax[i].set_xlabel(f"{features[i]}")
plt.tight_layout()
fig.savefig("outputs/scaled_hist.png", dpi=300)
logger.info(f"Figure saved to - outputs/scaled_hist.png")
plt.show()

# 5. Get statistics for each feature
logger.info("Calculated the mean, median, std and rms of each feature - saved to df")
X_scaled[features].agg(['mean', 'median', 'std', rms])

In [ ]:
# 5. sns.pairplot to visualize pairwise relationships between features and identify potential clusters or outliers in the data.
sns.pairplot(X_scaled, corner=True)
logger.info("Created corner pairplot via seaborn")

In [ ]:
# 6. Find the right number of clusters
logger.info("Finding the right number of clusters necessary for model")
inertia = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    inertia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

# Save elbow plot
plt.figure(figsize=(8,5))
plt.plot(list(K_range), inertia, 'bo-')
plt.xlabel('Number of clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.savefig("outputs/elbow_method.png", dpi=300)
logger.info(f"Elbow plot saved - outputs/elbow_method.png")
plt.show()

# Save silhouette plot
plt.figure(figsize=(8,5))
plt.plot(list(K_range), silhouette_scores, 'ro-')
plt.xlabel('Number of clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Scores')
plt.savefig("outputs/silhouette_scores.png", dpi=300)
logger.info("Silhouette score plot saved - outputs/silhouette_scores.png")
plt.show()

In [ ]:
# 7. Choose the K value and fit model
# Choosing silhouette score peak
logger.info("Choosing k value from the silhouette_scores and fitting the kmeans model")
optimal_k = np.argmax(silhouette_scores) + 2
logger.info(f"Best K (by silhouette): {optimal_k}")
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init="auto")
df['cluster'] = kmeans.fit_predict(X_scaled)

In [ ]:
# 8. Group data frame based on cluster column and take the mean of each feature per group
logger.info("Grouping rows based on cluster column and calculating the mean of each feature")
k_groups = df.groupby("cluster")
for i, grp in enumerate(k_groups):
    features_mean = grp[1][features].mean()
    logger.info(f"Cluster {i} means: {features_mean}")

In [ ]:
# 9. Obtain cluster centers and plot their scatter diagrams
centers_original = scaler.inverse_transform(kmeans.cluster_centers_)
centers_df = pd.DataFrame(centers_original, columns=features)
centers_df.to_csv("outputs/cluster_centers.csv", index=False)
logger.info("Cluster Centers saved to - outputs/cluster_centers.csv")
centers_df

plt.figure(figsize=(8,6))
sns.scatterplot(x=X_scaled[features[0]], y=X_scaled[features[1]], hue=df['cluster'], palette='Set1', s=100)
plt.scatter(kmeans.cluster_centers_[:,0], kmeans.cluster_centers_[:,1], s=300, c='yellow', marker='X', label='Centroids')
plt.xlabel(features[0])
plt.ylabel(features[1])
plt.legend()
plt.savefig("outputs/cluster_scatter.png", dpi=300)
logger.info("Cluster centers scatter diagram saved to - outputs/cluster_scatter.png")
plt.show()

In [ ]:
# 10. PCA plt
logger.info("Compressing the features down to 2 axes via PCA")
pca = PCA(n_components=2, random_state=42)
pca_data = pca.fit_transform(X_scaled)
df['PCA1'] = pca_data[:, 0]
df['PCA2'] = pca_data[:, 1]

plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x='PCA1', y='PCA2', hue='cluster', palette='Set2', s=100)
plt.savefig("outputs/pca_clusters.png", dpi=300)
logger.info("PCA figure saved to - outputs/pca_clusters.png")
plt.show()